In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# OpenQASM 3 at ang Qiskit SDK

<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay binuo gamit ang mga sumusunod na kinakailangan.
Inirerekomenda namin ang paggamit ng mga bersyong ito o mas bago.

```
qiskit[all]~=2.3.0
```
</details>

Ang Qiskit SDK ay nagbibigay ng ilang mga kasangkapan para sa pag-convert sa pagitan ng mga OpenQASM na representasyon ng mga quantum na programa, at ng klase na QuantumCircuit. Tandaan na ang mga kasangkapang ito ay nasa exploratory na yugto pa ng pagpapaunlad at patuloy na mag-eevolve habang lumalaki ang suporta ng Qiskit para sa dynamic na Circuit na kakayahan na ipinahayag ng OpenQASM 3.

> **Note:** Ang function na ito ay nasa exploratory na yugto pa. Kaya naman, malamang na mag-eevolve ang syntax at mga kakayahan nito.

## Mag-import ng OpenQASM 3 na programa sa Qiskit
Kailangan mong i-install ang package na `qiskit_qasm3_import ` para magamit ang function na ito. I-install gamit ang sumusunod na command.

In [1]:
import qiskit.qasm3

program = """
    OPENQASM 3.0;
    include "stdgates.inc";

    input float[64] a;
    qubit[3] q;
    bit[2] mid;
    bit[3] out;

    let aliased = q[0:1];

    gate my_gate(a) c, t {
      gphase(a / 2);
      ry(a) c;
      cx c, t;
    }
    gate my_phase(a) c {
      ctrl @ inv @ gphase(a) c;
    }

    my_gate(a * 2) aliased[0], q[{1, 2}][0];
    measure q[0] -> mid[0];
    measure q[1] -> mid[1];

    while (mid == "00") {
      reset q[0];
      reset q[1];
      my_gate(a) q[0], q[1];
      my_phase(a - pi/2) q[1];
      mid[0] = measure q[0];
      mid[1] = measure q[1];
    }

    if (mid[0]) {
      let inner_alias = q[{0, 1}];
      reset inner_alias;
    }

    out = measure q;
"""
circuit = qiskit.qasm3.loads(program)
circuit.draw("mpl")

<Image src="../docs/images/guides/interoperate-qiskit-qasm3/extracted-outputs/e805197c-51fb-4216-8d24-ae26633d29bc-0.svg" alt="Output of the previous code cell" />

Sa kasalukuyan, dalawang high-level na function ang available para sa pag-import mula sa OpenQASM 3 papasok sa Qiskit. Ang mga function na ito ay `load()`, na tumatanggap ng file name, at `loads()`, na tumatanggap ng programa mismo bilang string:

In [2]:
from qiskit import QuantumCircuit
from qiskit.qasm3 import dumps

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

dumps(qc)

'OPENQASM 3.0;\ninclude "stdgates.inc";\nbit[2] meas;\nqubit[2] q;\nh q[0];\ncx q[0], q[1];\nbarrier q[0], q[1];\nmeas[0] = measure q[0];\nmeas[1] = measure q[1];\n'

Sa halimbawang ito, nagde-define tayo ng quantum na programa gamit ang OpenQASM 3, at ginagamit ang `loads()` para direktang i-convert ito sa isang QuantumCircuit:

In [3]:
from qiskit import QuantumCircuit
from qiskit.qasm3 import dump

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

f = open("my_file.txt", "w")
dump(qc, f)
f.close()

![Output of the previous code cell](../docs/images/guides/interoperate-qiskit-qasm3/extracted-outputs/e805197c-51fb-4216-8d24-ae26633d29bc-0.svg)

## I-export sa OpenQASM 3
Maaari mong i-export ang Qiskit code sa OpenQASM 3 gamit ang `dumps()`, na nag-e-export sa string, o `dump()`, na nag-e-export sa file.

### Halimbawa gamit ang `dumps()`